# Model Training — XGBoost Churn Classifier
We train an XGBoost classifier on the preprocessed Telco dataset.
XGBoost is the industry standard for tabular business data and is
widely used on BCG X client engagements. We use MLflow to log
metrics — mirroring production ML practices.

In [ ]:
import sys
sys.path.append('..')
from src.preprocess import load_and_clean, encode_features, split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import pandas as pd

## Load and preprocess data
Using the preprocessing module from src/ to keep the pipeline
clean and reusable — the same functions will be called by the
dashboard and the FACET analysis in Sprint 2.

In [ ]:
df = load_and_clean('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df = encode_features(df)
X_train, X_test, y_train, y_test = split(df)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Features:     {X_train.shape[1]}")

## Train with MLflow experiment tracking
MLflow logs every run's parameters and metrics automatically.
This is standard practice on BCG X engagements — every experiment
is reproducible and comparable.

In [ ]:
with mlflow.start_run(run_name="rf_baseline"):
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    auc   = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    mlflow.log_metric("roc_auc", auc)
    print(f"ROC-AUC: {auc:.3f}")
    print(classification_report(y_test, preds))

## Results interpretation
A ROC-AUC above 0.80 means the model reliably ranks at-risk customers
above low-risk ones — giving the retention team a prioritised call list.
The classification report shows precision and recall per class,
which determines how we set the decision threshold in production.